# K_𝖖-algebras — a researcher's quick start

**What this is.** A small, dependency-free Python library that computes the
**K-theoretic Coulomb branch algebra** `A_𝖖[T]` of a 4d 𝒩=2 theory `T` — the
algebra of half-BPS line defects, following Ambrosino–Gaiotto (DESY-25-035).
Give it a theory and it computes:

- **structure constants** — the product `L_a · L_b` of canonical line defects;
- the **Schur index** `I_{a,b}(𝖖)` of a pair of line defects;
- the automorphism `ρ` and the `ρ²`-twisted **trace**;
- and it **checks the orthonormality conjecture** `I_{a,b} = δ_{a,b} + O(𝖖)`.

**Who it's for.** Anyone who wants to *compute with* these algebras without
reading the source. This page runs the **real library in your browser** (via
WebAssembly) — nothing to install. Everything below is copy-pasteable Python.

**What it can't do (yet).** It covers the theories with a known BPS-quiver or
cone/abelianized description (Argyres–Douglas, SQED/SU(2)/pure-U(N) gauge
theories, class-S A₁ / skein, and more); very large ranks can be slow, and some
families are still being added. When something isn't available, the library
**says so** rather than guessing.

This guide is organised by **task**. Jump to whichever you need.

### Running this

- **In this page**, nothing runs until you press **▶ Run all cells** (or a
  single cell's **Run**) — the Python kernel loads in the background first.
- **On your own machine**: the package is pure Python 3 (no dependencies).
  Put its `src/` layers on the path and `import` by name — the setup cell below
  is exactly what's needed. See the package `README.md`.

In [1]:
# --- One-time setup: put the library on the import path --------------------
# The modules import one another by bare name; add every src/<layer>/ dir to
# sys.path (this mirrors the package's own run_tests.py). On this page the
# library is already unpacked into the working directory.
import sys, pathlib
_here = pathlib.Path.cwd()
for _base in (_here, *_here.parents):
    if (_base / "src" / "core").is_dir():
        sys.path[:0] = [str(p) for p in (_base / "src").rglob("*")
                        if p.is_dir() and p.name != "__pycache__"]
        break
print("library ready")

library ready

## Task 1 — Compute a theory's structure constants and Schur index

A theory is specified by its **BPS quiver**: an antisymmetric integer **Dirac
pairing** `B` and the **node charges**. Here is the simplest interesting one —
the `A₂` **Argyres–Douglas** theory (the "pentagon"). From it the library
discovers the canonical line-defect basis and computes with it.

In [2]:
from bps_kalgebra import BPSKAlgebra

# A_q([A1, A2]) — the A2 Argyres-Douglas theory, from its BPS quiver.
T = BPSKAlgebra(pairing=[[0, 1], [-1, 0]], node_charges=[(1, 0), (0, 1)])

# Structure constants: the product of two canonical line defects.
print("L_a · L_b        :", T.multiply((1, 0), (0, 1)))        # (q)·L_(1,1)

# The Schur index of a pair of line defects, as a power series in q.
print("Schur index I_ab :", T.inner_product((1, 0), (1, 0), K=8))

L_a · L_b        : (q)*L_(1, 1)
Schur index I_ab : 1 - q^2 + q^4 + q^6 + O(q^9)

## Task 2 — Check the orthonormality conjecture on a theory

The central conjecture of the subject: the canonical line-defect basis is
**orthonormal for the Schur pairing to leading order**,
`I_{a,b}(𝖖) = δ_{a,b} + O(𝖖)`. You can test it on any theory — here it is,
both as a one-call axiom check and read directly off the `𝖖⁰` term.

In [3]:
# Run the full axiom battery (unital / multiplicative / bar-invariant /
# orthonormal) to order q^6:
print("axioms to q^6 :", T.verify_canonical_basis(K=6))

# ...and see orthonormality directly — the q^0 coefficient of I_ab is delta_ab:
print("I_aa[q^0] =", T.inner_product((1, 0), (1, 0), K=8)[0], " (diagonal -> 1)")
print("I_ab[q^0] =", T.inner_product((1, 0), (0, 1), K=8)[0], " (off-diagonal -> 0)")

axioms to q^6 : {'unital': True, 'multiplicative': True, 'bar_invariant': True, 'orthonormality': True}
I_aa[q^0] = 1  (diagonal -> 1)
I_ab[q^0] = 0  (off-diagonal -> 0)

## Task 3 — Plug in *your* theory (from a BPS quiver)

Any 4d 𝒩=2 theory with a BPS quiver works: give its Dirac pairing and node
charges. **Flavour symmetries are detected automatically** (from `ker B`), and
their fugacities `μ` appear in the trace. The example is the "hexagon" — a
`U(1)`-flavoured Argyres–Douglas theory (`ker B = Z·(1,1,1)`).

In [4]:
# One U(1) flavour symmetry (ker B = Z*(1,1,1)); its characters mu^{±k} appear
# in the trace, shown as [(±k,)].
H = BPSKAlgebra(pairing=[[0, 1, -1], [-1, 0, 1], [1, -1, 0]],
                node_charges=[(1, 0, 0), (0, 1, 0), (0, 0, 1)])
print("flavour ring    :", H.coefficient_ring())        # R(U(1)) = Z[mu^±1]
print("flavoured trace :", H.trace((0, 0, 0), K=4))

# You don't even need a chamber's spectrum generator — the library can build it
# from the quiver alone with build_S=True:
Tb = BPSKAlgebra(pairing=[[0, 1], [-1, 0]], node_charges=[(1, 0), (0, 1)],
                 build_S=True)
print("spec-free build :", Tb.multiply((1, 0), (0, 1)))

flavour ring    : AbelianZPlusRing(rank=1)
flavoured trace : 1 + ([(-1,)] + 1 + [(1,)])*q^2 + (2*[(-1,)] + [(-2,)] + 3 + 2*[(1,)] + [(2,)])*q^4 + O(q^5)
spec-free build : (q)*L_(1, 1)

## Task 4 — Find your theory

Beyond a bare quiver, many **named families** have direct constructors. A rough
map from physics to code:

| theory | how to build it |
|---|---|
| `A₂` Argyres–Douglas (pentagon) | `BPSKAlgebra([[0,1],[-1,0]], [(1,0),(0,1)])` |
| SQED with `N_f` hypers (`U(1)+N_f`) | `SQEDNfSampleKAlgebra(N_f)` — flavour `SU(N_f)` |
| pure `U(N)` gauge theory | `PureUNKAlgebra(N, default_rays(N))` |
| skein algebra of a surface (class-S A₁) | `SkeinKAlgebra.polygon(n)` |
| **any** BPS-quiver theory | `BPSKAlgebra(pairing, node_charges)` |

A few of these, computed:

In [5]:
# SQED with N_f flavours: a Lagrangian gauge theory, flavour symmetry SU(N_f).
from samples import SQEDNfSampleKAlgebra
print("SQED_2 flavour ring :", SQEDNfSampleKAlgebra(2).coefficient_ring())   # R(SU(2))

# Pure U(2) gauge theory: Wilson lines fuse as reps (fund (x) fund = Sym^2 (+) det),
# and its canonical basis is orthonormal (the conjecture, on a gauge theory):
from pure_un_kalgebra import PureUNKAlgebra, default_rays
U2 = PureUNKAlgebra(2, default_rays(2), max_len=2, K=8)
W = ((0, 0), (1, 0))                                        # the fundamental Wilson line
print("U(2) Wilson fusion  :", U2.multiply(W, W))           # L_(2,0) + L_(1,1)
print("U(2) orthonormal    :", U2.verify_orthonormality(((1, 0), (0, 0)),
                                                         ((1, 0), (0, 0)), 4))

# The skein algebra of a surface (class-S A1). The pentagon is the SAME algebra
# as Task 1, now built from curves on a surface — same Schur index:
from skein_kalgebra import PentagonSkeinKAlgebra
print("skein Schur index   :", PentagonSkeinKAlgebra().inner_product((1, 0), (1, 0), K=6))

SQED_2 flavour ring : SUNZPlusRing(2)
U(2) Wilson fusion  : L_((0, 0), (1, 1)) + L_((0, 0), (2, 0))
U(2) orthonormal    : True
skein Schur index   : 1 - q^2 + q^4 + q^6 + O(q^7)

## Where to go next

- **Change the numbers above.** Edit any Dirac pairing / node charges / labels
  and press **Run** — it is the real library, so you are computing real
  structure constants and Schur indices.
- **Full package + docs.** Per-topic guides live in `docs/` (the contract, the
  cone / RG / BPS / abelianized / skein realisations, and how the axioms
  determine the trace). Install is nothing — pure Python 3.
- **No-code tool.** A companion web applet lets you draw a BPS quiver and read
  off its data without writing Python.

**The two conjectures this library exists to test — and to *build* algebras
from:** orthonormality of the canonical basis, `I_{a,b} = δ_{a,b} + O(𝖖)`
(Task 2), and RG-flow intertwining, `F·S = X_γ + O(𝖖)`. All arithmetic is
exact in `Z[𝖖^±]`; the `O(𝖖)` is applied only when a result is read out.